In [ ]:
from corus import load_lenta
from collections import Counter
import os
import torch
import pandas as pd
import requests

random_state = 42

LENTA_URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
LOCAL_FILE = "lenta-ru-news.csv.gz"

# 1. Загрузка и подготовка датасета Lenta.Ru
def download_file(url, path):
    if not os.path.exists(path):
        print("Скачивание датасета Lenta.Ru...")
        response = requests.get(url, stream=True)
        with open(path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:
                    f.write(chunk)
        print("Датасет загружен.")

def load_lenta_dataset(path=LOCAL_FILE, sample_size=10000):
    download_file(LENTA_URL, path)
    df = pd.read_csv(path, compression='gzip')

    # Выборка и балансировка
    df = df.sample(n=min(sample_size, len(df)), random_state=random_state)
    topic_counts = df["topic"].value_counts()
    valid_topics = topic_counts[topic_counts >= 1000].index
    df = df[df["topic"].isin(valid_topics)]

    min_class_size = min(Counter(df["topic"]).values())
    df_balanced = df.groupby("topic").apply(
        lambda x: x.sample(min_class_size, random_state=random_state)
    ).reset_index(drop=True)

    return df_balanced

In [ ]:
df_balanced = load_lenta_dataset(path=LOCAL_FILE, sample_size=10000)

<ipython-input-5-aed1e9e89eef>:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby("topic").apply(


In [ ]:
from natasha import (
    Segmenter,
    NewsEmbedding,
    NewsMorphTagger,
    MorphVocab,
    Doc
)
import nltk
import re

nltk.download("stopwords")
from nltk.corpus import stopwords
russian_stopwords = set(stopwords.words("russian"))

# Инициализация компонентов
segmenter = Segmenter()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
morph_vocab = MorphVocab()

def preprocess_natasha(text):
    text = text.lower()
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    lemmas = []
    for token in doc.tokens:
        if token.text in russian_stopwords or len(token.text) <= 2:
            continue
        token.lemmatize(morph_vocab)
        lemma = token.lemma
        if re.match(r"[а-яё]+$", lemma):
            lemmas.append(lemma)
    return ' '.join(lemmas)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df_balanced['processed_text'] = df_balanced['text'].apply(preprocess_natasha)

Пайплайн с BERTopic, где:

Энкодер — быстрый и многозадачный MiniLM.

UMAP позволяет визуализировать скрытую структуру.

HDBSCAN находит нестабильные и нерегулярные кластеры.

CountVectorizer с биграммами даёт хорошие интерпретации.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords

# Эмбеддер
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# UMAP — снижение размерности
umap_model = UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=42)

# Кластеризация
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', prediction_data=True)

# Токенизация
russian_stopwords = stopwords.words("russian")

vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=russian_stopwords,
    min_df=5
)

# Модель BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True
)

# === 4. Обучение модели ===
topics, probs = topic_model.fit_transform(df_balanced["processed_text"])

# === 5. Визуализация тем ===
topic_model.visualize_topics()

2025-04-29 15:56:07,971 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/101 [00:00<?, ?it/s]

2025-04-29 16:02:35,892 - BERTopic - Embedding - Completed ✓
2025-04-29 16:02:35,895 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-29 16:02:55,142 - BERTopic - Dimensionality - Completed ✓
2025-04-29 16:02:55,144 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-29 16:02:55,561 - BERTopic - Cluster - Completed ✓
2025-04-29 16:02:55,566 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-04-29 16:02:56,683 - BERTopic - Representation - Completed ✓


In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))  # Топ-10 тем

# Токены для конкретной темы (например, Topic 1)
topic_model.get_topic(1)

   Topic  Count                               Name  \
0     -1      3  -1_зимний_комбинат_путин_общество   
1      0   2604      0_год_который_россия_сообщать   
2      1    266         1_год_центр_россия_который   
3      2    196      2_год_который_объявить_россия   
4      3    135         3_год_который_суд_сообщать   

                                      Representation  \
0  [зимний, комбинат, путин, общество, президент,...   
1  [год, который, россия, сообщать, человек, стра...   
2  [год, центр, россия, который, человек, сообщат...   
3  [год, который, объявить, россия, страна, сообщ...   
4  [год, который, суд, сообщать, человек, заявить...   

                                 Representative_Docs  
0  [минута молчание отметить американец собраться...  
1  [эксперт польский комиссия расследование авиак...  
2  [комитет сенат сша разведка заслушать объяснен...  
3  [закрытый часть список магнитский подготовить ...  
4  [президиум адвокатский палата москва стать лиш...  


[('год', np.float64(0.10659544393880836)),
 ('центр', np.float64(0.08555183349549124)),
 ('россия', np.float64(0.08255165023211791)),
 ('который', np.float64(0.07617733018297101)),
 ('человек', np.float64(0.06177006602351088)),
 ('сообщать', np.float64(0.06070133968725065)),
 ('время', np.float64(0.054603299025214935)),
 ('российский', np.float64(0.053765599217923093)),
 ('страна', np.float64(0.04509413672760083)),
 ('это', np.float64(0.04502189534392514))]

 топ-10 токенов по всем темам:

In [ ]:
topics_tokens = {
    topic: [word for word, _ in topic_model.get_topic(topic)[:10]]
    for topic in topic_model.get_topics().keys() if topic != -1
}

# Преобразуем в DataFrame
import pandas as pd
df_tokens = pd.DataFrame.from_dict(topics_tokens, orient='index')
df_tokens.columns = [f"Token_{i+1}" for i in range(df_tokens.shape[1])]
df_tokens.head()


,Token_1,Token_2,Token_3,Token_4,Token_5,Token_6,Token_7,Token_8,Token_9,Token_10
0,год,который,россия,сообщать,человек,страна,заявить,время,российский,свой
1,год,центр,россия,который,человек,сообщать,время,российский,страна,это
2,год,который,объявить,россия,страна,сообщать,свой,заявить,российский,дело
3,год,который,суд,сообщать,человек,заявить,слово,дело,решение,результат


In [ ]:
# Частотность топиков
topic_model.visualize_barchart(top_n_topics=10)


In [ ]:
def calculate_topic_diversity(topic_model):
    topic_diversity = []
    for topic in topic_model.get_topics().keys():
        if topic == -1:
            continue
        top_tokens = [token for token, _ in topic_model.get_topic(topic)]
        topic_diversity.append(len(set(top_tokens)))
    return sum(topic_diversity) / len(topic_diversity)

# Вычисление
diversity_score = calculate_topic_diversity(topic_model)
print(f"Среднее разнообразие тем: {diversity_score}")


Среднее разнообразие тем: 10.0
